In [3]:
import gradio as gr
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib

matplotlib.use("TkAgg")

from preprocesare import redimensionare, Imagini, centrare_date
from matematica import svd


# Fractia din centrul imaginii pe care o pastram (taie marginile cu fundal).
# 0.7 = pastram centrul 70%, aruncam 15% din fiecare margine.
FRACTIE_CROP = 0.7


def cropare_centrala(img_array, fractie=FRACTIE_CROP):
    """Pastreaza doar regiunea centrala a imaginii, unde se afla capul,
    eliminand marginile (fundal, umeri, perete) care strica recunoasterea."""
    h, w = img_array.shape
    nh, nw = int(h * fractie), int(w * fractie)
    sus = (h - nh) // 2
    stanga = (w - nw) // 2
    return img_array[sus:sus + nh, stanga:stanga + nw]


def calculeaza_sosia_dupa_selectie(imagine_de_la_interfata, k_componente):
    if imagine_de_la_interfata is None:
        return None

    img = Image.fromarray(imagine_de_la_interfata).convert("L")
    IMG = redimensionare(np.array(img), linii, coloane)

    v_tu = IMG.flatten().astype(np.float64)
    if np.linalg.norm(v_tu) == 0:
        return None
    v_tu_normat = v_tu / np.linalg.norm(v_tu)
    v_tu_centrat = v_tu_normat - Fata_medie

    k = int(k_componente)
    U_redus = U[:, :k]
    W_redus = W[:k, :]

    w_tu = U_redus.T @ v_tu_centrat
    distante = np.linalg.norm(W_redus - w_tu[:, np.newaxis], axis=0)
    index_minim = np.argmin(distante)

    sosia_vector = X[:, index_minim]
    sosia_matrice = sosia_vector.reshape(linii, coloane)
    sosia_0_255 = (
        (sosia_matrice - sosia_matrice.min())
        / (sosia_matrice.max() - sosia_matrice.min())
        * 255
    ).astype(np.uint8)

    return sosia_0_255

Fete, n, linii, coloane, target, target_names = Imagini()

baza_de_date = []
def construieste_amprenta_faciala(imagine_din_interfata, nr_componente=50):
    if imagine_din_interfata is None:
        return None, None
    img = Image.fromarray(imagine_din_interfata).convert("L")
    img_decupata = cropare_centrala(np.array(img))
    img_redimensionata = redimensionare(img_decupata, linii, coloane)
    v_fata = img_redimensionata.flatten().astype(np.float64)
    if np.linalg.norm(v_fata) == 0:
        return None, None
    v_Fata_normat = v_fata / np.linalg.norm(v_fata)
    v_fata_centrat = v_Fata_normat - Fata_medie
    U_redus = U[:, :nr_componente]
    w_amprenta = U_redus.T @ v_fata_centrat
    return w_amprenta, v_fata_centrat

def register(poza_fata, poza_dreapta, poza_stanga):
    global baza_de_date
    if poza_fata is None or poza_dreapta is None or poza_stanga is None:
        return "All three photos are required to enroll!"
    w_fata, _ = construieste_amprenta_faciala(poza_fata, nr_componente=50)
    w_dreapta, _ = construieste_amprenta_faciala(poza_dreapta, nr_componente=50)
    w_stanga, _ = construieste_amprenta_faciala(poza_stanga, nr_componente=50)
    baza_de_date.append((w_fata, w_dreapta, w_stanga))
    return "Enrollment successful!"

def login(poza_noua_camera):
    if poza_noua_camera is None:
        return "A new photo is required to log in!"
    if len(baza_de_date) == 0:
        return "The database is empty! Please enroll first."

    w_poza_camera, v_centrat = construieste_amprenta_faciala(poza_noua_camera, nr_componente=50)
    if w_poza_camera is None:
        return "Error processing the camera photo!"

    # Eroare de reconstructie: masoara cat de bine se incadreaza imaginea in spatiul fetelor
    U_redus = U[:, :50]
    reconstructie = U_redus @ w_poza_camera
    eroare = np.linalg.norm(v_centrat - reconstructie)
    if eroare > 0.85:
        return f"Rejected: doesn't look like a face (reconstruction error: {eroare:.3f})"

    # Comparare identitate
    w_salvat_fata, w_salvat_dreapta, w_salvat_stanga = baza_de_date[-1]
    distanta_fata = np.linalg.norm(w_poza_camera - w_salvat_fata)
    distanta_dreapta = np.linalg.norm(w_poza_camera - w_salvat_dreapta)
    distanta_stanga = np.linalg.norm(w_poza_camera - w_salvat_stanga)
    distanta_medie = np.mean([distanta_fata, distanta_dreapta, distanta_stanga])

    if distanta_medie < 0.35:
        return f"Authentication successful! (distance: {distanta_medie:.3f}, rec. error: {eroare:.3f})"
    else:
        return f"Authentication failed! (distance: {distanta_medie:.3f}, rec. error: {eroare:.3f})"


MAX_PER_PERSOANA = 5
selectati = []
contor = {}
for i in range(n):
    persoana = target[i]
    if persoana not in contor:
        contor[persoana] = 0
    if contor[persoana] < MAX_PER_PERSOANA:
        selectati.append(i)
        contor[persoana] += 1
    if len(selectati) >= 250:
        break

ok = 0
X_list = []
target_selectati = []
for i in selectati:
    if linii > 200 or coloane > 200:
        img_prelucrata = redimensionare(Fete[i], 200, 200)
        ok = 1
    else:
        img_prelucrata = Fete[i]
    v_brut = img_prelucrata.flatten().astype(np.float64)
    if np.linalg.norm(v_brut) == 0:
        continue
    v_normat = v_brut / np.linalg.norm(v_brut)
    X_list.append(v_normat)
    target_selectati.append(target[i])

X = np.array(X_list).T
if ok == 1:
    linii, coloane = 200, 200
A, Fata_medie = centrare_date(X)

"""
plt.imshow(Fata_medie.reshape(linii, coloane), cmap='gray')
plt.show()
"""

U, S, Vt = svd(A)

"""
Afișăm prima Eigenface (cea mai importantă)
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(U[:, 0].reshape(linii, coloane), cmap='gray')
plt.title("Eigenface 1 (Cea mai mare valoare proprie)")

Afișăm a doua Eigenface (următoarea ca importanță)
plt.subplot(1, 2, 2)
plt.imshow(U[:, 1].reshape(linii, coloane), cmap='gray')
plt.title("Eigenface 2")
plt.show()
"""

W = U.T @ A
"""
# 1. Extragem ponderile primei fețe (prima coloană din W)
w1 = W[:, 0] 
# 2. Reconstruim fața în spațiul pixelilor
# Înmulțim matricea U (Eigenfaces) cu vectorul de ponderi w1
fata_reconstruita_centrata =  U[:, :100] @ w1
# 3. Adăugăm înapoi Fața Medie (psi) pentru a reveni la aspectul original
fata_finala = fata_reconstruita_centrata + Fata_medie.flatten()
# 4. Afișăm rezultatul
plt.imshow(fata_finala.reshape(linii, coloane), cmap='gray')
plt.title("Prima față reconstruită din ponderile W")
plt.show() 
"""

custom_theme = gr.themes.Monochrome(
    primary_hue="slate",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
    radius_size=gr.themes.sizes.radius_lg,
).set(
    body_background_fill="#000000",
    body_background_fill_dark="#000000",
    body_text_color="white",
    body_text_color_dark="white",
    background_fill_primary="#050505",
    background_fill_primary_dark="#050505",
    background_fill_secondary="#0a0a0a",
    background_fill_secondary_dark="#0a0a0a",
    block_background_fill="rgba(255, 255, 255, 0.03)",
    block_background_fill_dark="rgba(255, 255, 255, 0.03)",
    block_border_width="1px",
    block_border_color="rgba(255, 255, 255, 0.08)",
    block_border_color_dark="rgba(255, 255, 255, 0.08)",
    button_primary_background_fill="white",
    button_primary_background_fill_dark="white",
    button_primary_text_color="black",
    button_primary_text_color_dark="black",
    shadow_drop="0px 4px 20px rgba(0, 0, 0, 0.5)",
)

custom_css = """
body, .gradio-container {
    background: radial-gradient(circle at 50% 20%, #152220 0%, #000000 70%) !important;
    color: #ffffff !important;
}
.tab-nav {
    background: rgba(255, 255, 255, 0.05) !important;
    border: 1px solid rgba(255, 255, 255, 0.1) !important;
    border-radius: 9999px !important;
    padding: 6px !important;
    display: flex;
    justify-content: center;
    gap: 10px;
    margin: 0 auto 30px auto !important;
    width: fit-content !important;
}
.tab-nav button {
    border-radius: 9999px !important;
    border: none !important;
    padding: 8px 24px !important;
    font-weight: 500 !important;
    color: #aaaaaa !important;
    transition: all 0.3s ease;
}
.tab-nav button.selected {
    background: rgba(255, 255, 255, 0.1) !important;
    color: white !important;
}
.gr-box {
    backdrop-filter: blur(15px);
}
h1 {
    text-align: center;
    font-size: 3.5em !important;
    font-weight: 600 !important;
    letter-spacing: -0.03em;
    margin-top: 1em !important;
    margin-bottom: 0.5em !important;
}
p {
    text-align: center;
    color: #aaaaaa;
}

/* Ghidaj de pozitionare a capului: DOAR un inel oval subtire.
   Fundalul din afara ovalului ramane complet transparent, ca sa NU acopere
   butoanele de webcam/upload. pointer-events:none => nimic nu e blocat. */
.ghidaj-fata {
    position: relative !important;
}
.ghidaj-fata::after {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    right: 0;
    bottom: 0;
    pointer-events: none;
    z-index: 20;
    border-radius: inherit;
    background: radial-gradient(
        ellipse 31% 38% at 50% 52%,
        rgba(255, 255, 255, 0) 0%,
        rgba(255, 255, 255, 0) 93%,
        rgba(255, 255, 255, 0.55) 94.5%,
        rgba(255, 255, 255, 0.55) 95.5%,
        rgba(255, 255, 255, 0) 97%
    );
}
"""

with gr.Blocks(theme=custom_theme, css=custom_css, title="Face Recognition with Eigenfaces (SVD)") as app_autentificare:
    gr.Markdown("# Face Recognition with Eigenfaces")
    
    with gr.Tab("1. Find Your Look-Alike (1:N)"):
        gr.Markdown("Upload a photo and adjust the number of principal components (k) to find the most similar face in the public database.")
        with gr.Row():
            with gr.Column():
                img_sosie_in = gr.Image(label="1. Choose / drop your photo here")
                k_slider = gr.Slider(minimum=1, maximum=150, value=15, step=1, label="Number of principal components (k)")
                btn_sosie = gr.Button("Find Look-Alike", variant="primary")
            with gr.Column():
                img_sosie_out = gr.Image(label="2. Your computed look-alike", image_mode="L")
        
        btn_sosie.click(fn=calculeaza_sosia_dupa_selectie, inputs=[img_sosie_in, k_slider], outputs=img_sosie_out)
        
    with gr.Tab("2. Enroll"):
        gr.Markdown("Before using the security system, the algorithm needs to learn your face. Take **3 captures (front, left, right)**. **Move close to the camera so your face fills the oval** — this removes the background, which hurts recognition.")
        
        with gr.Row():
            poza1 = gr.Image(label="Front view", elem_classes="ghidaj-fata")
            poza2 = gr.Image(label="Left view", elem_classes="ghidaj-fata")
            poza3 = gr.Image(label="Right view", elem_classes="ghidaj-fata")
            
        btn_inreg = gr.Button("Enroll me in the system!", variant="primary")
        rez_inreg = gr.Textbox(label="Enrollment status")
        
        btn_inreg.click(fn=register, inputs=[poza1, poza2, poza3], outputs=rez_inreg)
        
    with gr.Tab("3. Log In"):
        gr.Markdown("Upload a photo to test whether the security gate recognizes you. **Move close so your face fills the oval**, just like at enrollment.")
        
        poza_login = gr.Image(label="Security camera", elem_classes="ghidaj-fata")
        btn_login = gr.Button("Verify identity", variant="primary")
        rez_login = gr.Textbox(label="System decision")
        
        btn_login.click(fn=login, inputs=poza_login, outputs=rez_login)

app_autentificare.launch()

/tmp/ipykernel_213334/2654764208.py:284: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css, title="Face Recognition with Eigenfaces (SVD)") as app_autentificare:


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
